# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh/Content Opportunity Scoring**

I checked real signal strength for all four predefined lanes on the starter dataset before choosing (see the code cell below):

- **Ranking Signal Analysis** position and organic sessions correlate weakly in isolation (r ~ -0.045 across the 28,795 rows with a valid `avg_position`), so position alone isn't a strong standalone signal here.
- **Structured Content Archetype Clustering** `content_type` is heavily skewed (keyword article is ~90.7% of rows), leaving too little category diversity for a meaningful cluster story right now.
- **CTR / Engagement Opportunity Scoring** a real opportunity exists (7,491 of 15,082 visible, high-impression rows sit below the median CTR for their peer group), but it answers a narrower question than refresh, and folds naturally into refresh as a review flag rather than standing alone.
- **Refresh / Content Opportunity Scoring** the target (`trend_direction` -> `is_declining_label`) is directly observed in trailing performance, not hand-defined, and more than half the catalog (54.2%) is currently trending down: a large, real, actionable labeled set.

I'm picking **Refresh/Content Opportunity Scoring** because it has the clearest observed target, the largest usable positive class, and it maps onto a decision an editor already has to make every week.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # work/notebooks/ -> repo root

import pandas as pd

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), \
    "starter CSV not found -- are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows:", len(df), "| clients:", df['client_id'].nunique())

# Lane 1: Ranking Signal Analysis
valid_pos = df[df['avg_position'] > 0]
print("\nLane 1 (Ranking Signal): valid-position rows:", len(valid_pos),
      "| corr(avg_position, sessions_90d):", round(valid_pos['avg_position'].corr(valid_pos['sessions_90d']), 3))

# Lane 3: Archetype Clustering
print("\nLane 3 (Archetype Clustering) content_type share:")
print((df['content_type'].value_counts(normalize=True) * 100).round(1))

# Lane 4: CTR / Engagement Opportunity
visible = df[(df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['impressions_90d'] > 100)]
low_ctr = visible[visible['ctr'] < visible['ctr'].median()]
print("\nLane 4 (CTR Opportunity): visible+high-impression rows:", len(visible),
      "| below-median-CTR opportunity rows:", len(low_ctr))

# Lane 2: Refresh / Content Opportunity (chosen lane)
trend_counts = df['trend_direction'].value_counts()
declining_share = (df['trend_direction'] == 'down').mean()
print("\nLane 2 (Refresh, chosen):")
print(trend_counts)
print("declining share:", round(declining_share, 3))

rows: 30000 | clients: 32

Lane 1 (Ranking Signal): valid-position rows: 28795 | corr(avg_position, sessions_90d): -0.045

Lane 3 (Archetype Clustering) content_type share:
content_type
keyword article       90.7
feedly article         7.0
comparison article     2.3
Name: proportion, dtype: float64

Lane 4 (CTR Opportunity): visible+high-impression rows: 15082 | below-median-CTR opportunity rows: 7491

Lane 2 (Refresh, chosen):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
declining share: 0.542


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** given a fixed weekly review budget, which existing content pages should the content/SEO editor prioritize for a refresh review first, out of thousands of live pages.

**Who acts:** a content/SEO editor with limited bandwidth, realistically able to deep-review only a few dozen pages a week the reference pipeline's ranked queue is sized around a top-50 cut for exactly this reason.

**Cost of a wrong call:**
- *False positive* (flagged as needing refresh but it's actually fine): wasted editor hours, and a real decliner further down the queue waits a week longer than it should.
- *False negative* (a genuinely declining page ranked low or missed): the page keeps losing organic sessions silently, and for transactional/commercial-intent pages that also means lost revenue-adjacent clicks (see the numbers below).

Because review time is the scarce resource, the practical cost isn't "being wrong" in the abstract it's spending editor hours on the wrong pages while the real decliners keep sliding.

In [3]:
declining = df[df['trend_direction'] == 'down']
commercial_declining = declining[declining['main_intent'].isin(['transactional', 'commercial'])]

print("declining pages total:", len(declining))
print("declining AND transactional/commercial intent (revenue-adjacent):", len(commercial_declining),
      f"({len(commercial_declining) / len(declining):.1%} of all decliners)")

weekly_review_capacity = 50
backlog_weeks = len(declining) / weekly_review_capacity
print(f"\nat a {weekly_review_capacity}-page/week review capacity, clearing the current declining "
      f"backlog alone would take ~{backlog_weeks:.0f} weeks — prioritization is not optional, it's required.")

declining pages total: 16262
declining AND transactional/commercial intent (revenue-adjacent): 5652 (34.8% of all decliners)

at a 50-page/week review capacity, clearing the current declining backlog alone would take ~325 weeks — prioritization is not optional, it's required.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
print(f"1) {len(df):,} content items across {df['client_id'].nunique()} clients, "
      f"{declining_share:.1%} of them currently trending down (trend_direction == 'down').")

valid_pos_rate = (df['avg_position'] > 0).mean()
print(f"2) avg_position is usable (non-zero) for {valid_pos_rate:.1%} of rows "
      f"({(df['avg_position']==0).sum():,} rows have avg_position == 0, meaning 'no data', not rank zero).")

opp_rate = len(low_ctr) / len(visible)
print(f"3) among visible, high-impression pages (position <= 20, impressions_90d > 100), "
      f"{opp_rate:.1%} sit below the median CTR for their peer group -- a concrete refresh-worthy slice, "
      f"not a marginal one.")

1) 30,000 content items across 32 clients, 54.2% of them currently trending down (trend_direction == 'down').
2) avg_position is usable (non-zero) for 96.0% of rows (1,205 rows have avg_position == 0, meaning 'no data', not rank zero).
3) among visible, high-impression pages (position <= 20, impressions_90d > 100), 49.7% sit below the median CTR for their peer group -- a concrete refresh-worthy slice, not a marginal one.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** observed/measured associations between trailing signals (visibility, content age, CTR, engagement) and content that is *currently* in a downward performance trend, on this anonymized sample. A decision-support ranking that reorders the same catalog an editor already owns in line with the reference pipeline's demonstrated lift over a hand-written rule (Precision@50 0.24 -> 0.74 on the bundled sample).

**What I can never claim:**
- That refreshing a flagged page *causes* it to recover there's no pre/post experiment or causal design here, only trailing correlation.
- That the model "predicts Google's algorithm" I never observe ranking mechanics, only downstream traffic outcomes.
- That `trend_direction` / `trend_pct` are legitimate predictive *features* — `is_declining_label` is derived directly from them, so using them as inputs would leak the label into the model rather than predict anything new (demonstrated below).

In [5]:
# Leakage check: trend_pct trivially separates trend_direction because the label IS derived from it.
leak_check = df.groupby('trend_direction')['trend_pct'].agg(['min', 'max', 'count'])
print(leak_check)
print("\n-> trend_direction == 'down' maps almost perfectly onto negative trend_pct ranges.")
print("   trend_direction / trend_pct must be EXCLUDED from model features -- they define the label, not predict it.")

                   min      max  count
trend_direction                       
down            -100.0    -20.0  16262
flat               NaN      NaN      0
new                NaN      NaN      0
stable           -20.0     20.0   5962
up                20.0  44900.0   4388

-> trend_direction == 'down' maps almost perfectly onto negative trend_pct ranges.
   trend_direction / trend_pct must be EXCLUDED from model features -- they define the label, not predict it.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.